# STAR Door-State Classifier - Google Colab Training

Fine-tunes **YOLOv8n-cls** on the **DeepDoors2** dataset (gasparramoa/DeepDoors2) to classify a rectified camera frame as `open`, `closed`, or `ajar`.
Outputs `door_state_yolov8n.onnx` ready to drop into `star-compliance/star_compliance/models/`.

**Dataset:** 3000 RGB images, 1000 per class (closed / open / semi-open), 480x640, from https://github.com/gasparramoa/DeepDoors2

**Runtime:** Runtime -> Change runtime type -> Hardware accelerator: GPU (T4 is fine, ~15-20 min end to end).

**Prereq:** Add a shortcut to the DeepDoors 2 folder (https://drive.google.com/drive/folders/1SxVKeJ9RBcoJXHSHw-LWaLGG07BZT-b5) into your personal My Drive before running section 4.

**Sections:**
1. Verify GPU
2. Install dependencies
3. Paths and hyperparameters
4. Mount Drive and locate the DeepDoors2 dataset
5. Normalize dataset layout (`Semi-open` -> `ajar`, drop `Depth/`, use `RGB/`)
6. Train YOLOv8n-cls
7. Validate on held-out test split
8. Export to ONNX (opset 12, simplified)
9. md5 + size report
10. Download the ONNX file to your local machine

## 1. Verify GPU

In [33]:
!nvidia-smi

Sat Apr 18 05:53:26 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   45C    P8             14W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 2. Install dependencies

In [34]:
!pip install --quiet \
    'ultralytics>=8.3.0' \
    'onnx>=1.16' \
    'onnxruntime>=1.17' \
    'onnxsim>=0.4.35' \
    'opencv-python>=4.9'

## 3. Paths and hyperparameters

Tweak `EPOCHS`, `IMG_SIZE`, `BATCH_SIZE` here if you want to experiment.

In [35]:
import os

WORK_DIR = '/content/door_state_training'
DATASET_RAW = f'{WORK_DIR}/dataset-raw'
DATASET_ROOT = f'{WORK_DIR}/dataset'
RUNS_DIR = f'{WORK_DIR}/runs'
OUTPUT_ONNX = f'{WORK_DIR}/door_state_yolov8n.onnx'

EPOCHS = 50
IMG_SIZE = 320
BATCH_SIZE = 64
DEVICE = 0

os.makedirs(WORK_DIR, exist_ok=True)
os.makedirs(DATASET_ROOT, exist_ok=True)
os.makedirs(RUNS_DIR, exist_ok=True)
%cd {WORK_DIR}

/content/door_state_training


## 4. Mount Drive and locate the DeepDoors2 dataset

**One-time setup (do this in a browser first):**
1. Open https://drive.google.com/drive/folders/1SxVKeJ9RBcoJXHSHw-LWaLGG07BZT-b5
2. Right-click the `DeepDoors 2` folder -> **Organize -> Add shortcut** -> anywhere under your My Drive

The cell below mounts Drive and recursively searches MyDrive for a folder whose name matches DeepDoors2 (with or without a space), then symlinks it in as `dataset-raw/`. No file copying - faster and saves Colab disk.

In [36]:
from google.colab import drive
from pathlib import Path

drive.mount('/content/drive')

SPLIT_NAMES_LC = {'train', 'val', 'test'}

def looks_like_deepdoors2(path):
    """Accept only if the folder has an RGB subtree with train/val/test (any case)."""
    for root, dirs, _ in os.walk(str(path), followlinks=True):
        if os.path.basename(root).lower() == 'rgb':
            child_names = {d.lower() for d in os.listdir(root)}
            if SPLIT_NAMES_LC.issubset(child_names):
                return True
        depth = root.count(os.sep) - str(path).count(os.sep)
        if depth > 5:
            dirs[:] = []
    return False

candidates = []
search_roots = ['/content/drive/MyDrive']
for base in search_roots:
    for p in Path(base).rglob('*'):
        if not p.is_dir():
            continue
        name = p.name.lower().replace(' ', '').replace('-', '').replace('_', '')
        if name in {'deepdoors2', 'deepdoors'} and looks_like_deepdoors2(p):
            candidates.append(p)

if not candidates:
    raise RuntimeError(
        'Could not find a DeepDoors 2 folder with an RGB/ tree under MyDrive. '
        'In Drive: Shared with me -> right-click DeepDoors 2 -> Organize -> '
        'Add shortcut -> My Drive (root). Then re-run this cell.'
    )

print('Found valid DeepDoors folders:')
for c in candidates:
    print(f'  {c}')

source = candidates[0]
print(f'\nUsing: {source}')

if os.path.islink(DATASET_RAW):
    os.remove(DATASET_RAW)
elif os.path.isdir(DATASET_RAW):
    import shutil as _sh
    _sh.rmtree(DATASET_RAW)

os.symlink(str(source), DATASET_RAW)
print(f'Linked: {DATASET_RAW} -> {source}')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Found valid DeepDoors folders:
  /content/drive/MyDrive/Personal/3dsbackup/Nintendo 3DS/4fa0bda4b88b4a421f978e1385d9ed0a/b0a201084738b2b9534530340002544d/title/0004000e/00086300/content/cmd/DeepDoors 2

Using: /content/drive/MyDrive/Personal/3dsbackup/Nintendo 3DS/4fa0bda4b88b4a421f978e1385d9ed0a/b0a201084738b2b9534530340002544d/title/0004000e/00086300/content/cmd/DeepDoors 2
Linked: /content/door_state_training/dataset-raw -> /content/drive/MyDrive/Personal/3dsbackup/Nintendo 3DS/4fa0bda4b88b4a421f978e1385d9ed0a/b0a201084738b2b9534530340002544d/title/0004000e/00086300/content/cmd/DeepDoors 2


## 5. Normalize dataset layout

YOLOv8 classification expects:

```
dataset/
  train/{open,closed,ajar}/*.jpg
  val/{open,closed,ajar}/*.jpg
  test/{open,closed,ajar}/*.jpg
```

DeepDoors2 ships with both `RGB/` and `Depth/` modalities under `Door Classification/`. We keep only `RGB/` and rename `Semi-open` -> `ajar`.

Source layout after download: `.../Door Classification/RGB/{Test,Train,Val}/{Closed,Open,Semi-open}/*.jpg`

In [ ]:
import shutil
import os
from pathlib import Path

# Clear any partial dataset/ from previous runs.
if os.path.isdir(DATASET_ROOT):
    shutil.rmtree(DATASET_ROOT)
os.makedirs(DATASET_ROOT, exist_ok=True)

rgb_root = None
for root, dirs, _ in os.walk(DATASET_RAW, followlinks=True):
    if os.path.basename(root).lower() == 'rgb':
        child_names = {d.lower() for d in os.listdir(root)}
        if {'train', 'val', 'test'}.issubset(child_names):
            rgb_root = root
            break

if rgb_root is None:
    raise RuntimeError(f'RGB/ tree not found under {DATASET_RAW}.')

print(f'RGB root: {rgb_root}')

CLASS_DST = {
    'closed': 'closed',
    'open': 'open',
    'semi': 'ajar',
    'semi-open': 'ajar',
    'semiopen': 'ajar',
}
SPLIT_DST = {'train': 'train', 'val': 'val', 'test': 'test'}
IMG_EXTS = {'.jpg', '.jpeg', '.png'}

total_copied = 0
for split_entry in sorted(os.listdir(rgb_root)):
    split_lc = split_entry.lower()
    if split_lc not in SPLIT_DST:
        continue
    split_src = os.path.join(rgb_root, split_entry)
    split_dst_name = SPLIT_DST[split_lc]

    for cls_entry in sorted(os.listdir(split_src)):
        cls_src = os.path.join(split_src, cls_entry)
        if not os.path.isdir(cls_src):
            continue
        cls_key = cls_entry.lower().replace('-', '').replace(' ', '')
        if cls_key not in CLASS_DST:
            continue
        cls_dst_name = CLASS_DST[cls_key]

        dst_dir = os.path.join(DATASET_ROOT, split_dst_name, cls_dst_name)
        os.makedirs(dst_dir, exist_ok=True)

        entries = os.listdir(cls_src)
        print(f'  {split_entry}/{cls_entry}: copying {len(entries)} files -> {split_dst_name}/{cls_dst_name} ...', flush=True)

        copied_here = 0
        for i, name in enumerate(entries):
            src_path = os.path.join(cls_src, name)
            if not os.path.isfile(src_path):
                continue
            ext = os.path.splitext(name)[1].lower()
            dst_name = name if ext in IMG_EXTS else name + '.jpg'
            shutil.copy2(src_path, os.path.join(dst_dir, dst_name))
            copied_here += 1
            if (i + 1) % 100 == 0:
                print(f'    {i + 1}/{len(entries)}', flush=True)

        print(f'    done: {copied_here} copied', flush=True)
        total_copied += copied_here

print(f'\nTotal images copied: {total_copied}')
print('\nImages per split/class:')
for split in ['train', 'val', 'test']:
    for cls in ['open', 'closed', 'ajar']:
        count = sum(1 for _ in Path(DATASET_ROOT, split, cls).glob('*'))
        print(f'  {split:<5} / {cls:<7}  {count:4d} images')

In [37]:
import shutil
import os
from pathlib import Path

# Clear any partial dataset/ from previous runs.
if os.path.isdir(DATASET_ROOT):
    shutil.rmtree(DATASET_ROOT)
os.makedirs(DATASET_ROOT, exist_ok=True)

rgb_root = None
for root, dirs, _ in os.walk(DATASET_RAW, followlinks=True):
    if os.path.basename(root).lower() == 'rgb':
        child_names = {d.lower() for d in os.listdir(root)}
        if {'train', 'val', 'test'}.issubset(child_names):
            rgb_root = root
            break

if rgb_root is None:
    raise RuntimeError(f'RGB/ tree not found under {DATASET_RAW}.')

print(f'RGB root: {rgb_root}')

CLASS_DST = {
    'closed': 'closed',
    'open': 'open',
    'semi': 'ajar',
    'semi-open': 'ajar',
    'semiopen': 'ajar',
}
SPLIT_DST = {'train': 'train', 'val': 'val', 'test': 'test'}
IMG_EXTS = {'.jpg', '.jpeg', '.png'}

def is_jpeg_magic(path):
    """Return True if the first two bytes are FF D8 (JPEG SOI marker)."""
    try:
        with open(path, 'rb') as f:
            return f.read(2) == b'\xff\xd8'
    except OSError:
        return False

total_copied = 0
for split_entry in os.listdir(rgb_root):
    split_lc = split_entry.lower()
    if split_lc not in SPLIT_DST:
        continue
    split_src = os.path.join(rgb_root, split_entry)
    split_dst_name = SPLIT_DST[split_lc]

    for cls_entry in os.listdir(split_src):
        cls_src = os.path.join(split_src, cls_entry)
        if not os.path.isdir(cls_src):
            continue
        cls_key = cls_entry.lower().replace('-', '').replace(' ', '')
        if cls_key not in CLASS_DST:
            continue
        cls_dst_name = CLASS_DST[cls_key]

        dst_dir = os.path.join(DATASET_ROOT, split_dst_name, cls_dst_name)
        os.makedirs(dst_dir, exist_ok=True)

        copied_here = 0
        for name in os.listdir(cls_src):
            src_path = os.path.join(cls_src, name)
            if not os.path.isfile(src_path):
                continue
            ext = os.path.splitext(name)[1].lower()
            if ext in IMG_EXTS:
                dst_name = name
            elif is_jpeg_magic(src_path):
                dst_name = name + '.jpg'
            else:
                continue
            shutil.copy2(src_path, os.path.join(dst_dir, dst_name))
            copied_here += 1
            total_copied += 1
        print(f'  {split_entry}/{cls_entry} -> {split_dst_name}/{cls_dst_name}: {copied_here}')

print(f'\nTotal images copied: {total_copied}')
print('\nImages per split/class:')
for split in ['train', 'val', 'test']:
    for cls in ['open', 'closed', 'ajar']:
        count = sum(1 for _ in Path(DATASET_ROOT, split, cls).glob('*'))
        print(f'  {split:<5} / {cls:<7}  {count:4d} images')

RGB root: /content/door_state_training/dataset-raw/Door Classification/RGB
  test/Open -> test/open: 0
  test/Semi -> test/ajar: 0
  test/Closed -> test/closed: 0


KeyboardInterrupt: 

## 6. Train YOLOv8n-cls

Metrics to watch in the per-epoch output:

| Metric | Target |
|---|---|
| top1 accuracy | >= 0.85 |
| top1 (open) | >= 0.90 |
| top1 (closed) | >= 0.90 |
| top1 (ajar) | >= 0.80 |

DeepDoors2 is well balanced (1000 per class), so ajar should train much better than on the smaller DoorDetect-Class-Dataset.

In [ ]:
!yolo classify train \
    model=yolov8n-cls.pt \
    data={DATASET_ROOT} \
    epochs={EPOCHS} \
    imgsz={IMG_SIZE} \
    batch={BATCH_SIZE} \
    device={DEVICE} \
    patience=15 \
    project={RUNS_DIR} \
    name=door_state_yolov8n \
    exist_ok=true \
    save_period=10

## 7. Validate on held-out test split

Acceptance thresholds for the capstone:
- Overall top-1 accuracy: >= 0.85
- Per-class top-1: open >= 0.90, closed >= 0.90, ajar >= 0.80

In [ ]:
BEST_PT = f'{RUNS_DIR}/door_state_yolov8n/weights/best.pt'
assert os.path.isfile(BEST_PT), f'Expected best.pt at {BEST_PT}'

!yolo classify val \
    model={BEST_PT} \
    data={DATASET_ROOT} \
    imgsz={IMG_SIZE} \
    split=test \
    device={DEVICE} \
    project={RUNS_DIR} \
    name=door_state_eval \
    exist_ok=true

## 8. Export to ONNX

Opset 12 with simplification; matches what `door_state_classifier.py` expects at runtime.

In [ ]:
!yolo export \
    model={BEST_PT} \
    format=onnx \
    imgsz={IMG_SIZE} \
    opset=12 \
    simplify=true

BEST_ONNX = f'{RUNS_DIR}/door_state_yolov8n/weights/best.onnx'
assert os.path.isfile(BEST_ONNX), f'ONNX export failed; expected {BEST_ONNX}'

shutil.copy2(BEST_ONNX, OUTPUT_ONNX)
print(f'Copied: {OUTPUT_ONNX}')

## 9. md5 + size report

Record these in `star-compliance/star_compliance/models/README.md` alongside the trained weights.

In [ ]:
import hashlib

def md5_of(path):
    h = hashlib.md5()
    with open(path, 'rb') as f:
        for chunk in iter(lambda: f.read(1 << 20), b''):
            h.update(chunk)
    return h.hexdigest()

size_bytes = os.path.getsize(OUTPUT_ONNX)
digest = md5_of(OUTPUT_ONNX)
print(f'ONNX:  {OUTPUT_ONNX}')
print(f'Size:  {size_bytes} bytes')
print(f'md5:   {digest}')

## 10. Download ONNX to your local machine

Drop the downloaded file into
`star-compliance/star_compliance/models/door_state_yolov8n.onnx`
in the STAR repo, update `models/README.md` with the md5 + dataset citation
(DeepDoors2, gasparramoa), run `pytest tests/` in the compliance-engine
package, then commit and push.

In [ ]:
from google.colab import files
files.download(OUTPUT_ONNX)